<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Voice cloning, speaker checks and misuse

**[Voice cloning, speaker checks and misuse](https://learning.nextia-ai.com/courses/speech/m09/voice-cloning-speaker-checks-and-misuse/)** · Speech AI: Transcribe, Understand and Speak · Module 9, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** enrol a voice from three turns of a call, score other speech against it, and count false accepts and false rejects at each threshold. You will see why a speaker check can be one signal, but never the only key to an account.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells and the speaker model (27 MB, downloaded once). No account and no key. |
| **You should know** | Speaker embeddings and the cosine similarity, from [Speaker embeddings and clustering](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0, matplotlib 3.11.2 and sherpa-onnx 2.0.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/voice-cloning-speaker-checks-and-misuse/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M09-L03-voice-cloning-speaker-checks-and-misuse/voice-cloning-speaker-checks-and-misuse-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, sherpa-onnx 2.0.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 sherpa-onnx==2.0.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'sherpa-onnx']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 22 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/calls/phone/C-0225.wav": "0eab5a465ce8234b1c6e8354f7447cdaa5434eef6d7de6ccfb09ad0978d699b0",
    "data/calls/phone/C-0144.wav": "4d0e80b1dba9a9b12372ca31e047550fe50ef6df4b62b8fc41bd5bd0561976ce",
    "data/calls/phone/C-0061.wav": "b29537eb1908e11ae85d1fd0334525eced36ba88f4f2424804b98d953c62087c",
    "data/calls/phone/C-0083.wav": "8896c2f5abf5d1fc71248906bfd6341e778efc4bf6acf24cd0115e000ef90e24",
    "data/calls/phone/C-0123.wav": "65b32c5561c0c44e22bfbda79ac9dba46be599ba5d38970d8f73558407e707a5",
    "data/calls/phone/C-0127.wav": "5de4b6c7bac7e55d5ac89c3956fe47993ead6e2102d8c715f825947f3d0c34ac",
    "data/calls/phone/C-0161.wav": "a7daaee59326c613433abb195965c59808a27f3f37c0ff12fd901e02074d7b4d",
    "data/calls/phone/C-0189.wav": "505cae3c15a73d04444b57bfa00814e60fb0abc99649a63c9cec7a27aa33e75d",
    "data/calls/phone/C-0192.wav": "384207262162543c3d505047116df079e0d238d36c22ec6f2704efb0db68a823",
    "data/calls/phone/C-0204.wav": "2fc187d2960de73f32beb554c90781e0a2853520dfce12582d47de0e5f6911e5",
    "data/calls/phone/C-0217.wav": "ee4b12fdb7a34aabc34f4bef71c04f3dd523c8d06f61b0c217240f5ae94fee7e",
    "data/calls/phone/C-0223.wav": "5c7267b27e9b8e5ac2ac03f5bcd80188ac62c4c83d767b61ff1f4c9aac0c1618",
    "data/calls/phone/C-0251.wav": "d6c47e1efc3fbff3041f918b6cc33ce32a940829d6d6407147afdd4ef73c8a17",
    "data/calls/phone/C-0288.wav": "65746af96f4920a6ddad457b87ac6eca3b7a34183389d1863db5509308e33da4",
    "data/calls/phone/C-0003.wav": "e03c761cb765170c39bcc4d676e3f4f7144f510c38ae476182d4cb5432792761",
    "data/calls/phone/C-0011.wav": "638e502682c2aad19a69eb50725adf2eebd7db209a54f592b5b7ff2736130747",
    "data/calls/phone/C-0044.wav": "2c9e0b36c65b1e013cd7f583768ad55ae077e13469a41060d93fcfc0e2f86b84",
    "data/calls/phone/C-0052.wav": "aef214c5c6b1a6358b6fd872e6325958b38480d2af92d8336266ff537b0fdc59",
    "data/calls/phone/C-0056.wav": "149f84cebaa075f01f4de38fe3918309c4bb37473cf1f2d11ac9d0521baf9deb",
    "data/calls/phone/C-0063.wav": "0b6ba00b04c8719109062a99f7434b5476fb7cd19e6aad8916e9cdfc8765ad00",
    "data/calls/phone/C-0066.wav": "c499be014b3a41584d8761a848b2f543023dad96705189aaa47b8ef82c265a1b",
    "data/calls/phone/C-0069.wav": "42975e806182e6539f3c785c64976e189ea03d33e2d769fc8bd0b4cecf658a0a",
    "data/calls/phone/C-0096.wav": "b22e0a9fcfc96ed41e4aaf0d72cb4c4ba6832b1cd357d122bf306cf950b893de",
    "data/calls/phone/C-0108.wav": "5cd79dfe3361933b72be1e901e46b799f83d09adfb45550e18dad623136ba2f4",
    "data/calls/phone/C-0126.wav": "4d4477775c593aeb9c3bf1b51e917861568efe21a69990d415f65c7982fa1b17",
    "data/calls/phone/C-0128.wav": "0b13d91f03204848d364c6ecf7adef4c05fc11ab68da9c226e543fa0cf95213a",
    "data/calls/phone/C-0133.wav": "23f155e7616b8ac85fba828a6898a8876f87a7eb9059de7566d4c62a88071687",
    "data/calls/phone/C-0196.wav": "eef024b1e2b613b7dae1d969b41bdb859fb4d4928aa9d39ae1272d55a8ee817f",
    "data/calls/phone/C-0207.wav": "ad965b3e8ff87ca8feaba5aacc965f84415474354db389236c431e45c0f405b5",
    "data/calls/phone/C-0219.wav": "cf505a17cbcd93cfeacfedd50dd424be352bf42676461b2b174124345b8e6172",
    "data/calls/phone/C-0265.wav": "3b2dc88b1627ca1feafff6a10d3c67bb74e9611cf41c4cb16193523ca093b9c2",
    "data/calls/phone/C-0275.wav": "ab0eafcc8b2f7712e8b47c6421d934839846a3a416a59f2fc12529af4a1bd8a0",
    "data/calls/phone/C-0279.wav": "9dc4f026fdef1c8353d4dbb16cede35576e580d44596c8a7b7503e830b2b5ce5",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The question: can a voice prove who is calling?

A bank clerk knows a regular customer's voice. Still, before she moves money, she asks for the account number and sends a code. A familiar voice is a hint, not proof.

**Speaker verification** answers: "Is this the person they say they are?" Enrol the person once (a **voiceprint**: the average of a few speaker embeddings), then compare new speech with it (the cosine similarity), and accept at a **threshold**. A **false reject** turns away the real person; a **false accept** lets in someone else.

Our "customer" is the agent Chloe: a synthetic voice (Piper, trained on LibriTTS-R, speaker 5401) who speaks in many calls. No real person's voice is used, and no voice is cloned.

## 2. The speaker model

The same model as in Module 5: WeSpeaker ResNet-34, through sherpa-onnx. The next two cells download it once and define `embed()`.

In [ ]:
import hashlib, subprocess, urllib.request
from pathlib import Path

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

In [ ]:
# The speaker model: WeSpeaker ResNet-34, trained on VoxCeleb (Apache-2.0), the sherpa-onnx copy (27 MB).
import sherpa_onnx
# Two places for the same file (the checksum is checked either way): GitHub, then a copy on Hugging Face.
MODEL_URLS = ["https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/"
              "wespeaker_en_voxceleb_resnet34_LM.onnx",
              "https://huggingface.co/csukuangfj/speaker-embedding-models/resolve/main/"
              "wespeaker_en_voxceleb_resnet34_LM.onnx"]
MODEL_SHA256 = "e9848563da86f263117134dfd7ad63c92355b37de492b55e325400c9d9c39012"
model_file = Path("models/wespeaker_en_voxceleb_resnet34_LM.onnx")
if not (model_file.exists() and sha256(model_file) == MODEL_SHA256):
    model_file.parent.mkdir(exist_ok=True)
    for url in MODEL_URLS:
        try:
            download(url, model_file)
        except SystemExit:
            continue  # this server did not answer: try the next one
        if model_file.exists() and sha256(model_file) == MODEL_SHA256:
            break
    else:
        raise SystemExit("Could not download the speaker model with the right checksum. Delete the folder models "
                         "and run again later.")
extractor = sherpa_onnx.SpeakerEmbeddingExtractor(
    sherpa_onnx.SpeakerEmbeddingExtractorConfig(model=str(model_file), num_threads=2))

def embed(x, sr=16000):
    """One piece of speech -> one speaker embedding, scaled to length 1."""
    stream = extractor.create_stream()
    stream.accept_waveform(sr, x)
    stream.input_finished()
    e = np.array(extractor.compute(stream))
    return e / np.linalg.norm(e)

print("Speaker model ready: each embedding has", extractor.dim, "numbers.")

## 3. Enrol Chloe from three turns of one call

Call C-0225 is stereo: Chloe is alone on the agent channel. The next cell cuts her three turns.

In [ ]:
calls = load_calls()
CHLOE, SIMILAR = "en_US-libritts_r-medium:5401", "en_US-libritts_r-medium:1914"

def turns_of(call_id, who, keep=lambda text: True, shortest=1.5):
    """(text, audio at 16 kHz) of each turn of `who` that lasts at least `shortest` seconds.
    Stereo calls: that speaker's own channel. Mono calls: the mix (turns do not overlap in our calls)."""
    c = calls[call_id]
    channel = (0 if who == "agent" else 1) if c["channels"] == 2 else None
    x, sr = read_audio(f"data/calls/phone/{call_id}.wav", 16000, channel)
    return [(t["text"], x[int(t["start"] * sr):int(t["end"] * sr)]) for t in c["turns"]
            if t["speaker"] == who and t["end"] - t["start"] >= shortest and keep(t["text"])]

enrol = turns_of("C-0225", "agent")[:3]
for text, a in enrol:
    print(f"{len(a) / 16000:4.1f} s  {text}")

In [ ]:
voiceprint = np.mean([embed(a) for _, a in enrol], axis=0)
voiceprint /= np.linalg.norm(voiceprint)               # the average direction, length 1 again

def score(a):
    """Cosine similarity between a piece of speech and Chloe's voiceprint."""
    return float(embed(a) @ voiceprint)

print("Chloe's voiceprint:", voiceprint.shape, "numbers")

## 4. Score everybody

The next cell scores three groups against Chloe's voiceprint:

- **Chloe** herself: one turn in each other call where she is the agent, never the greeting, so the words differ from the enrolment.
- **The similar voice**: the caller of C-0144, the synthetic voice closest to Chloe that we found (speaker 1914 of the same model). It is not a clone.
- **Other voices**: the first turn of each English caller with a different voice, on the stereo calls.

> **Predict.** Will every turn of Chloe score higher than every other voice?

In [ ]:
not_greeting = lambda text: not text.startswith(("Thank you for calling", "Merci"))
with_audio = [c for c in calls.values() if c["split"] != "train"]
same = []                                               # Chloe, in other calls, other sentences
for c in sorted(with_audio, key=lambda c: c["call_id"]):
    if c["voices"]["agent"] == CHLOE and c["call_id"] not in ("C-0225", "C-0144"):
        text, a = turns_of(c["call_id"], "agent", not_greeting)[0]
        same.append(score(a))
similar = [score(a) for _, a in turns_of("C-0144", "caller")]
others, seen = [], set()                                # one turn of each other English caller voice
for c in sorted(with_audio, key=lambda c: c["call_id"]):
    v = c["voices"]["caller"]
    if c["language"] == "en" and c["channels"] == 2 and v not in seen and v not in (CHLOE, SIMILAR):
        seen.add(v)
        others.append(score(turns_of(c["call_id"], "caller")[0][1]))
print("Chloe     ", len(same), "turns:", np.round(sorted(same), 3))
print("similar   ", len(similar), "turns:", np.round(similar, 3))
print("others    ", len(others), "voices: median", round(float(np.median(others)), 3), " highest", round(max(others), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
for row, (name, values, colour) in enumerate([("Chloe", same, "tab:blue"), ("similar voice", similar, "tab:red"),
                                              ("other voices", others, "tab:gray")]):
    ax.scatter(values, [row] * len(values), color=colour, s=40, alpha=0.8)
ax.set_yticks([0, 1, 2], ["Chloe", "similar voice", "other voices"])
ax.set_xlabel("cosine similarity to Chloe's voiceprint"); ax.set_xlim(0.45, 1.0)
for t in (0.7, 0.8, 0.9):
    ax.axvline(t, color="black", lw=0.6, ls="--")
plt.tight_layout(); plt.show()

> **Check.** Chloe: 12 turns, from 0.751 up, median about 0.919. The similar voice: [0.846, 0.864]. The 28 other voices: median 0.711, highest 0.931. The groups overlap: some other voices score higher than some turns of Chloe, and the similar voice sits among Chloe's turns.

## 5. Your turn: count who gets in

> **Your turn.** Write `accepted(scores, threshold)`. It returns how many scores are at or above the threshold.

In [ ]:
def accepted(scores, threshold):
    """How many scores are at or above the threshold (the check says: this is Chloe)."""
    return sum(1 for s in scores if s >= threshold)

In [ ]:
table = {}
for t in (0.5, 0.6, 0.7, 0.8, 0.9):
    table[t] = (accepted(same, t), accepted(similar, t), accepted(others, t))
    print(f"threshold {t}: Chloe {table[t][0]} of {len(same)}, similar {table[t][1]} of {len(similar)}, "
          f"others {table[t][2]} of {len(others)}")

In [ ]:
expect_hash('table', table, '33677a125edf9e14')

> **Check.** At 0.7: Chloe 12 of 12, other voices 14 of 28. At 0.8: Chloe 11, the similar voice 2 of 2, other voices still 9. At 0.9: Chloe only 7 of 12, and 1 other voice still passes. No threshold accepts Chloe and refuses everybody else.

The lesson's measured run (short sentences made by the voices, through the phone line) gave the same picture: at 0.7, 19 of 30 other voices passed; at 0.8, Chloe passed only 1 of 3 sentences. Longer turns, as here, help a little. They do not make the voice a password.

## 6. Failure case: testing on the enrolment

A common mistake is to test the check with the same speech that enrolled it.

In [ ]:
own = [score(a) for _, a in enrol]
print("the enrolment turns themselves:", np.round(own, 3))

> **Check.** The enrolment turns score [0.94, 0.947, 0.946]: all three at 0.94 or more, higher than most of Chloe's other turns. A test on them alone would pass Chloe at a threshold of 0.9 every time, and hide the 5 of her 12 other turns that score below 0.9 (the lowest: 0.751). An attacker never says your enrolment sentence for you, and a real customer does not repeat it either. Always test with other sentences, other calls and other voices, and count false accepts and false rejects apart.

## Next

A speaker score can be one signal: it may send a doubtful call to a person. Risky actions, such as a refund or a new email address, need something the caller knows and something they have, such as a one-time code. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/voice-cloning-speaker-checks-and-misuse/) has the decision diagram, the rules for telling callers that they talk to a machine, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Watch the system](https://learning.nextia-ai.com/courses/speech/m09/watch-the-system/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m09/voice-cloning-speaker-checks-and-misuse/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.